In [ ]:
import pandas as pd
import json
import ast
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, accuracy_score

# 1. LOAD DATA (Fixes 'df' not defined)
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# Use your files: train.jsonl and test.jsonl
df = pd.concat([load_jsonl('train.jsonl'), load_jsonl('test.jsonl')], ignore_index=True)

# 2. EXTRACT LABELS (Fixes 'Cleaned Threat Description' KeyError)
def extract_label(entities):
    if not entities: return "benign"
    if isinstance(entities, str): entities = ast.literal_eval(entities)
    return entities[0].get('label', 'benign')

df['Threat Category'] = df['entities'].apply(extract_label)
df['text_data'] = df['text'].fillna('no description')

# 3. VECTORIZATION & SPLIT (Fixes 'tfidf' and 'X_train' not defined)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

# 1. IMPROVED VECTORIZER (Focus on technical patterns)
# Using a custom regex token pattern to keep hashes and filepaths intact
tfidf_improved = TfidfVectorizer(
    max_features=10000, 
    ngram_range=(1, 3), 
    token_pattern=r'[a-zA-Z0-9\\/:\._-]+' 
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


over = SMOTE(sampling_strategy='auto', k_neighbors=1, random_state=42)
under = RandomUnderSampler(sampling_strategy='auto', random_state=42)
X_res, y_res = over.fit_resample(X_train, y_train)
X_res, y_res = under.fit_resample(X_res, y_res)

# 2. DEFINING THE HIGH-PRECISION ENSEMBLE
# We use 'scale_pos_weight' logic inside XGBoost for the imbalanced classes
xgb_final = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    tree_method='hist',
    random_state=42
)

rf_final = RandomForestClassifier(
    n_estimators=400,
    class_weight='balanced_subsample', # Better for 20+ rare classes
    random_state=42
)

# 3. HYBRID VOTING
# We give RF more weight for category depth and XGB more weight for precision
final_model = VotingClassifier(
    estimators=[('xgb', xgb_final), ('rf', rf_final)],
    voting='soft',
    weights=[1, 2]
)

# 4. TRAINING (Run this on your X_res, y_res data)
print("Training High-Precision Hybrid Model...")
X_tfidf = tfidf_improved.fit_transform(df['text_data'])
# ... (Perform split and SMOTE as before)
final_model.fit(X_res, y_res)

# 5. EVALUATE
y_pred = final_model.predict(X_test)
print(f"Improved Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Training High-Precision Hybrid Model...
